# Protected area and forest reserve size

This notebook calculates the total area of Jamaica protected areas and forest reserves in hectares and square kilometres.

Two area measures are reported:
- **Feature-sum area**: sums every feature area. This can double-count where features overlap.
- **Dissolved area**: dissolves each layer first, then calculates area. This avoids double-counting within the same layer.

The notebook also reports the overlap between the protected-area layer and the forest-reserve layer, plus their combined dissolved area with no double-counting between layers.

## Imports

In [ ]:
from pathlib import Path

import geopandas as gpd
import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)

## Paths and settings

In [ ]:
base_path = Path("/Users/robynhaggis/Documents/Geospatial_analysis/dphil_papers")
jamaica_metric_grid_crs = "EPSG:3448"

protected_areas_path = base_path / "dphil_common_cross_cutting/common_incoming_data/protected_landcover/protected_areas.shp"
forest_reserves_path = base_path / "dphil_common_cross_cutting/common_incoming_data/protected_landcover/forest_reserves.shp"

output_dir = base_path / "dphil_paper_3/processed_data/baseline_assessment/protected_area_size"
output_dir.mkdir(parents=True, exist_ok=True)
summary_csv_path = output_dir / "protected_area_forest_reserve_size_summary.csv"
source_layer_summary_csv_path = output_dir / "protected_area_forest_reserve_source_layer_summary.csv"
funding_density_csv_path = output_dir / "protected_area_forest_reserve_usd7m_density.csv"

total_funding_usd = 7_000_000

for required_path in [protected_areas_path, forest_reserves_path]:
    if not required_path.exists():
        raise FileNotFoundError(f"Missing required input: {required_path}")

print(f"Protected areas: {protected_areas_path}")
print(f"Forest reserves: {forest_reserves_path}")
print(f"Outputs: {output_dir}")

## Load source layers

In [ ]:
def load_area_layer(vector_path, target_crs):
    area_layer = gpd.read_file(vector_path).to_crs(target_crs)
    area_layer = area_layer[area_layer.geometry.notna() & ~area_layer.geometry.is_empty].copy()
    area_layer["geometry"] = area_layer.geometry.make_valid()
    return area_layer


protected_areas = load_area_layer(protected_areas_path, jamaica_metric_grid_crs)
forest_reserves = load_area_layer(forest_reserves_path, jamaica_metric_grid_crs)

source_layer_summary = pd.DataFrame(
    [
        {
            "layer": "Protected areas",
            "source_path": str(protected_areas_path),
            "feature_count": len(protected_areas),
            "crs": str(protected_areas.crs),
            "geometry_types": ", ".join(sorted(protected_areas.geom_type.unique())),
        },
        {
            "layer": "Forest reserves",
            "source_path": str(forest_reserves_path),
            "feature_count": len(forest_reserves),
            "crs": str(forest_reserves.crs),
            "geometry_types": ", ".join(sorted(forest_reserves.geom_type.unique())),
        },
    ]
)

display(source_layer_summary)

## Calculate areas

In [ ]:
def summarise_area(layer_name, area_layer):
    feature_sum_m2 = area_layer.geometry.area.sum()
    dissolved_area_m2 = area_layer.geometry.union_all().area
    overlap_within_layer_m2 = feature_sum_m2 - dissolved_area_m2

    return {
        "summary_item": layer_name,
        "feature_count": len(area_layer),
        "feature_sum_area_ha": feature_sum_m2 / 10_000,
        "feature_sum_area_km2": feature_sum_m2 / 1_000_000,
        "dissolved_area_ha": dissolved_area_m2 / 10_000,
        "dissolved_area_km2": dissolved_area_m2 / 1_000_000,
        "within_layer_overlap_or_duplicate_area_ha": overlap_within_layer_m2 / 10_000,
        "within_layer_overlap_or_duplicate_area_km2": overlap_within_layer_m2 / 1_000_000,
    }


protected_areas_union = protected_areas.geometry.union_all()
forest_reserves_union = forest_reserves.geometry.union_all()
protected_forest_overlap_m2 = protected_areas_union.intersection(forest_reserves_union).area
combined_protected_forest_union_m2 = protected_areas_union.union(forest_reserves_union).area
combined_protected_forest_union_ha = combined_protected_forest_union_m2 / 10_000
combined_protected_forest_union_km2 = combined_protected_forest_union_m2 / 1_000_000
combined_protected_forest_usd_per_ha = total_funding_usd / combined_protected_forest_union_ha
combined_protected_forest_usd_per_km2 = total_funding_usd / combined_protected_forest_union_km2

area_summary = pd.DataFrame(
    [
        summarise_area("Protected areas", protected_areas),
        summarise_area("Forest reserves", forest_reserves),
        {
            "summary_item": "Protected areas ∩ forest reserves overlap",
            "feature_count": np.nan,
            "feature_sum_area_ha": np.nan,
            "feature_sum_area_km2": np.nan,
            "dissolved_area_ha": protected_forest_overlap_m2 / 10_000,
            "dissolved_area_km2": protected_forest_overlap_m2 / 1_000_000,
            "within_layer_overlap_or_duplicate_area_ha": np.nan,
            "within_layer_overlap_or_duplicate_area_km2": np.nan,
        },
        {
            "summary_item": "Protected areas ∪ forest reserves combined",
            "feature_count": np.nan,
            "feature_sum_area_ha": np.nan,
            "feature_sum_area_km2": np.nan,
            "dissolved_area_ha": combined_protected_forest_union_ha,
            "dissolved_area_km2": combined_protected_forest_union_km2,
            "within_layer_overlap_or_duplicate_area_ha": np.nan,
            "within_layer_overlap_or_duplicate_area_km2": np.nan,
        },
    ]
)

area_summary["usd_7m_per_ha"] = np.nan
area_summary["usd_7m_per_km2"] = np.nan
combined_area_mask = area_summary["summary_item"].eq("Protected areas ∪ forest reserves combined")
area_summary.loc[combined_area_mask, "usd_7m_per_ha"] = combined_protected_forest_usd_per_ha
area_summary.loc[combined_area_mask, "usd_7m_per_km2"] = combined_protected_forest_usd_per_km2

funding_density_summary = pd.DataFrame(
    [
        {
            "funding_scenario": "US$7m divided by combined protected-area + forest-reserve area",
            "total_funding_usd": total_funding_usd,
            "combined_no_double_count_area_ha": combined_protected_forest_union_ha,
            "combined_no_double_count_area_km2": combined_protected_forest_union_km2,
            "usd_per_ha": combined_protected_forest_usd_per_ha,
            "usd_per_km2": combined_protected_forest_usd_per_km2,
        }
    ]
)

area_summary_rounded = area_summary.copy()
numeric_summary_columns = area_summary_rounded.select_dtypes(include="number").columns
area_summary_rounded[numeric_summary_columns] = area_summary_rounded[numeric_summary_columns].round(2)

funding_density_summary_rounded = funding_density_summary.copy()
numeric_funding_columns = funding_density_summary_rounded.select_dtypes(include="number").columns
funding_density_summary_rounded[numeric_funding_columns] = funding_density_summary_rounded[numeric_funding_columns].round(2)

display(area_summary_rounded)
display(funding_density_summary_rounded)

## Save outputs

In [ ]:
area_summary_rounded.to_csv(summary_csv_path, index=False, float_format="%.2f")
source_layer_summary.to_csv(source_layer_summary_csv_path, index=False)
funding_density_summary_rounded.to_csv(funding_density_csv_path, index=False, float_format="%.2f")

print(f"Saved area summary: {summary_csv_path}")
print(f"Saved source layer summary: {source_layer_summary_csv_path}")
print(f"Saved funding density summary: {funding_density_csv_path}")

## Interpretation notes

- Use `dissolved_area_ha` / `dissolved_area_km2` when reporting total area without double-counting overlaps within a layer.
- Use `feature_sum_area_ha` / `feature_sum_area_km2` only if you intentionally want to sum all feature records as supplied.
- The combined protected-area + forest-reserve dissolved area removes overlap between the two layers.
- `usd_7m_per_ha` and `usd_7m_per_km2` divide US$7 million by the combined no-double-count area only.